In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    Dense
)

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)



In [4]:
df = pd.read_csv("./Dataset/df_for_EDA.csv")

In [5]:
df = df.set_index("Datetime")

In [6]:
df = df.sort_index()

print(df.index.min())
print(df.index.max())
print(df.shape)

1998-04-01 01:00:00
2015-01-01 00:00:00
(145198, 14)


In [7]:
len(df) * 0.20

29039.600000000002

In [8]:
test_size = int(len(df) * 0.20)

test_df = df.iloc[-test_size:].copy()

remaining_df = df.iloc[:-test_size].copy()

In [9]:
60 * 24

1440

In [10]:
validation_hours = 60 * 24

val_df = remaining_df.iloc[-validation_hours:].copy()

train_df = remaining_df.iloc[:-validation_hours].copy()

In [11]:
print("Train:")
print(train_df.index.min(), "→", train_df.index.max())
print(train_df.shape)

print("\nValidation:")
print(val_df.index.min(), "→", val_df.index.max())
print(val_df.shape)

print("\nTest:")
print(test_df.index.min(), "→", test_df.index.max())
print(test_df.shape)

Train:
1998-04-01 01:00:00 → 2011-05-11 03:00:00
(114719, 14)

Validation:
2011-05-11 04:00:00 → 2011-07-10 03:00:00
(1440, 14)

Test:
2011-07-10 04:00:00 → 2015-01-01 00:00:00
(29039, 14)


In [12]:
scaler = StandardScaler()

train_scaled = scaler.fit_transform( train_df[["PJME_MW"]] )


val_scaled = scaler.transform( val_df[["PJME_MW"]] )


test_scaled = scaler.transform( test_df[["PJME_MW"]] )

In [13]:
train_scaled

array([[-1.61544069],
       [-1.73285452],
       [-1.77132563],
       ...,
       [ 0.73114322],
       [ 0.55817711],
       [ 0.16961888]], shape=(114719, 1))

In [14]:
test_scaled

array([[-1.14255379],
       [-1.49387197],
       [-1.6449865 ],
       ...,
       [ 0.1556154 ],
       [ 0.12730066],
       [ 0.0665163 ]], shape=(29039, 1))

In [15]:
val_scaled

array([[-0.21693885],
       [-0.75784267],
       [-1.00713547],
       ...,
       [-0.26341195],
       [-0.4983935 ],
       [-0.80985561]], shape=(1440, 1))

In [16]:
def create_sequences(data, sequence_length, forecast_horizon):

    X = []
    y = []

    for i in range(sequence_length, len(data) - forecast_horizon + 1):

        X.append(data[i-sequence_length:i])

        y.append(data[i:i+forecast_horizon])

    return np.array(X), np.array(y)

In [17]:
SEQUENCE_LENGTH = 24
FORECAST_HORIZON = 24


X_train, y_train = create_sequences(
    train_scaled,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [18]:
X_train

array([[[-1.61544069],
        [-1.73285452],
        [-1.77132563],
        ...,
        [-0.254025  ],
        [-0.59257078],
        [-0.95466088]],

       [[-1.73285452],
        [-1.77132563],
        [-1.76363141],
        ...,
        [-0.59257078],
        [-0.95466088],
        [-1.20949352]],

       [[-1.77132563],
        [-1.76363141],
        [-1.67699446],
        ...,
        [-0.95466088],
        [-1.20949352],
        [-1.33844868]],

       ...,

       [[ 0.29980512],
        [-0.03412412],
        [-0.40806333],
        ...,
        [ 1.17617704],
        [ 0.90672537],
        [ 0.71883247]],

       [[-0.03412412],
        [-0.40806333],
        [-0.4776191 ],
        ...,
        [ 0.90672537],
        [ 0.71883247],
        [ 0.55879264]],

       [[-0.40806333],
        [-0.4776191 ],
        [-0.7476863 ],
        ...,
        [ 0.71883247],
        [ 0.55879264],
        [ 0.18916221]]], shape=(114672, 24, 1))

In [19]:
y_train

array([[[-1.20949352],
        [-1.33844868],
        [-1.38322906],
        ...,
        [-0.34758674],
        [-0.74537803],
        [-1.1354751 ]],

       [[-1.33844868],
        [-1.38322906],
        [-1.37507318],
        ...,
        [-0.74537803],
        [-1.1354751 ],
        [-1.40738892]],

       [[-1.38322906],
        [-1.37507318],
        [-1.27843375],
        ...,
        [-1.1354751 ],
        [-1.40738892],
        [-1.55465633]],

       ...,

       [[ 0.55879264],
        [ 0.18916221],
        [-0.23278894],
        ...,
        [ 0.68851723],
        [ 0.49893159],
        [ 0.73114322]],

       [[ 0.18916221],
        [-0.23278894],
        [-0.634889  ],
        ...,
        [ 0.49893159],
        [ 0.73114322],
        [ 0.55817711]],

       [[-0.23278894],
        [-0.634889  ],
        [-0.90634116],
        ...,
        [ 0.73114322],
        [ 0.55817711],
        [ 0.16961888]]], shape=(114672, 24, 1))

In [20]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (114672, 24, 1)
y_train shape: (114672, 24, 1)


In [21]:
val_input = np.concatenate([
    train_scaled[-SEQUENCE_LENGTH:],
    val_scaled
])


X_val, y_val = create_sequences(
    val_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [22]:
val_input

array([[-0.23278894],
       [-0.634889  ],
       [-0.90634116],
       ...,
       [-0.26341195],
       [-0.4983935 ],
       [-0.80985561]], shape=(1464, 1))

In [23]:
X_val

array([[[-0.23278894],
        [-0.634889  ],
        [-0.90634116],
        ...,
        [ 0.73114322],
        [ 0.55817711],
        [ 0.16961888]],

       [[-0.634889  ],
        [-0.90634116],
        [-1.09161803],
        ...,
        [ 0.55817711],
        [ 0.16961888],
        [-0.21693885]],

       [[-0.90634116],
        [-1.09161803],
        [-1.21195567],
        ...,
        [ 0.16961888],
        [-0.21693885],
        [-0.75784267]],

       ...,

       [[-0.48623663],
        [-0.87017832],
        [-1.22072708],
        ...,
        [-0.26725906],
        [-0.09783229],
        [-0.19785717]],

       [[-0.87017832],
        [-1.22072708],
        [-1.50679827],
        ...,
        [-0.09783229],
        [-0.19785717],
        [-0.47392587]],

       [[-1.22072708],
        [-1.50679827],
        [-1.64437096],
        ...,
        [-0.19785717],
        [-0.47392587],
        [-0.82801398]]], shape=(1417, 24, 1))

In [24]:
y_val

array([[[-0.21693885],
        [-0.75784267],
        [-1.00713547],
        ...,
        [ 1.17525373],
        [ 0.95335236],
        [ 0.4830815 ]],

       [[-0.75784267],
        [-1.00713547],
        [-1.1770239 ],
        ...,
        [ 0.95335236],
        [ 0.4830815 ],
        [-0.0027317 ]],

       [[-1.00713547],
        [-1.1770239 ],
        [-1.28674351],
        ...,
        [ 0.4830815 ],
        [-0.0027317 ],
        [-0.58579986]],

       ...,

       [[-0.47392587],
        [-0.82801398],
        [-1.17748555],
        ...,
        [-0.22324811],
        [-0.0719797 ],
        [-0.26341195]],

       [[-0.82801398],
        [-1.17748555],
        [-1.49017875],
        ...,
        [-0.0719797 ],
        [-0.26341195],
        [-0.4983935 ]],

       [[-1.17748555],
        [-1.49017875],
        [-1.64052385],
        ...,
        [-0.26341195],
        [-0.4983935 ],
        [-0.80985561]]], shape=(1417, 24, 1))

In [25]:
test_input = np.concatenate([
    val_scaled[-SEQUENCE_LENGTH:],
    test_scaled
])


X_test, y_test = create_sequences(
    test_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [26]:
test_input

array([[-1.17748555],
       [-1.49017875],
       [-1.64052385],
       ...,
       [ 0.1556154 ],
       [ 0.12730066],
       [ 0.0665163 ]], shape=(29063, 1))

In [27]:
X_test

array([[[-1.17748555],
        [-1.49017875],
        [-1.64052385],
        ...,
        [-0.26341195],
        [-0.4983935 ],
        [-0.80985561]],

       [[-1.49017875],
        [-1.64052385],
        [-1.72362145],
        ...,
        [-0.4983935 ],
        [-0.80985561],
        [-1.14255379]],

       [[-1.64052385],
        [-1.72362145],
        [-1.74162593],
        ...,
        [-0.80985561],
        [-1.14255379],
        [-1.49387197]],

       ...,

       [[-0.70506031],
        [-0.75630383],
        [ 0.02173593],
        ...,
        [-0.26525856],
        [-0.33327549],
        [-0.41083325]],

       [[-0.75630383],
        [ 0.02173593],
        [ 0.01281063],
        ...,
        [-0.33327549],
        [-0.41083325],
        [-0.43637807]],

       [[ 0.02173593],
        [ 0.01281063],
        [ 0.01219509],
        ...,
        [-0.41083325],
        [-0.43637807],
        [-0.41868135]]], shape=(29016, 24, 1))

In [28]:
y_test

array([[[-1.14255379],
        [-1.49387197],
        [-1.6449865 ],
        ...,
        [-0.86186856],
        [-1.01005928],
        [-1.21349451]],

       [[-1.49387197],
        [-1.6449865 ],
        [-1.72192872],
        ...,
        [-1.01005928],
        [-1.21349451],
        [-1.4470911 ]],

       [[-1.6449865 ],
        [-1.72192872],
        [-1.74239535],
        ...,
        [-1.21349451],
        [-1.4470911 ],
        [-1.44170515]],

       ...,

       [[-0.43637807],
        [-0.41868135],
        [-0.84817284],
        ...,
        [ 0.19562535],
        [ 0.18639229],
        [ 0.1556154 ]],

       [[-0.41868135],
        [-0.84817284],
        [-0.7990837 ],
        ...,
        [ 0.18639229],
        [ 0.1556154 ],
        [ 0.12730066]],

       [[-0.84817284],
        [-0.7990837 ],
        [-0.73245174],
        ...,
        [ 0.1556154 ],
        [ 0.12730066],
        [ 0.0665163 ]]], shape=(29016, 24, 1))

In [29]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (114672, 24, 1)
y_train: (114672, 24, 1)
X_val: (1417, 24, 1)
y_val: (1417, 24, 1)
X_test: (29016, 24, 1)
y_test: (29016, 24, 1)


# Basic RNN

In [30]:
def build_rnn_model(sequence_length):

    model = Sequential([
        
        tf.keras.layers.Input(
            shape=(sequence_length, 1)
        ),

        SimpleRNN(
            64,
            activation="tanh"
        ),

        Dense(64, activation="relu"),

        Dense(24)
    ])

    return model

In [31]:
rnn_model = build_rnn_model(
    SEQUENCE_LENGTH
)

rnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

E0000 00:00:1786424500.971224   93018 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [32]:
history_rnn = rnn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10


1792/1792 ━━━━━━━━━━━━━━━━━━━━ 25s 12ms/step - loss: 0.1640 - mae: 0.2931 - val_loss: 0.2211 - val_mae: 0.3452
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 23s 13ms/step - loss: 0.1302 - mae: 0.2607 - val_loss: 0.2144 - val_mae: 0.3376
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - loss: 0.1244 - mae: 0.2537 - val_loss: 0.2154 - val_mae: 0.3362
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - loss: 0.1198 - mae: 0.2481 - val_loss: 0.2564 - val_mae: 0.3783
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - loss: 0.1164 - mae: 0.2439 - val_loss: 0.2473 - val_mae: 0.3660
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - loss: 0.1131 - mae: 0.2402 - val_loss: 0.2227 - val_mae: 0.3456
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.1104 - mae: 0.2373 - val_loss: 0.2167 - val_mae: 0.3390
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.1076 - mae: 0.2344 - val_loss: 0.2219 - val_mae: 0.3389
Epoch 9/10
1792/1792 ━━━━━━━━━━━━━━

In [33]:
history_rnn = rnn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
    
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.1008 - mae: 0.2272 - val_loss: 0.2053 - val_mae: 0.3313
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.0988 - mae: 0.2249 - val_loss: 0.2024 - val_mae: 0.3314
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - loss: 0.0975 - mae: 0.2233 - val_loss: 0.1674 - val_mae: 0.2936
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - loss: 0.0959 - mae: 0.2215 - val_loss: 0.1932 - val_mae: 0.3208
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0943 - mae: 0.2195 - val_loss: 0.1846 - val_mae: 0.3171
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - loss: 0.0940 - mae: 0.2189 - val_loss: 0.1838 - val_mae: 0.3138
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - loss: 0.0924 - mae: 0.2172 - val_loss: 0.1666 - val_mae: 0.2986
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - loss: 0.0915 - mae: 0.2159 - val_loss: 0.1668 - val_mae: 0.2975
Epoch 9/10
1792/1792 ━━━━━━━━━

# LSTM

In [34]:
def build_lstm_model(sequence_length):

    model = Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, 1)
        ),

        LSTM(64),

        Dense(64, activation="relu"),

        Dense(24)
    ])

    return model

In [35]:
lstm_model = build_lstm_model(
    SEQUENCE_LENGTH
)

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

history_lstm = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 35s 18ms/step - loss: 0.1858 - mae: 0.3162 - val_loss: 0.2481 - val_mae: 0.3771
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - loss: 0.1299 - mae: 0.2618 - val_loss: 0.2307 - val_mae: 0.3563
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 34s 19ms/step - loss: 0.1215 - mae: 0.2506 - val_loss: 0.2404 - val_mae: 0.3659
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 38s 21ms/step - loss: 0.1169 - mae: 0.2449 - val_loss: 0.1964 - val_mae: 0.3238
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - loss: 0.1128 - mae: 0.2394 - val_loss: 0.2111 - val_mae: 0.3347
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - loss: 0.1091 - mae: 0.2347 - val_loss: 0.1813 - val_mae: 0.3090
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - loss: 0.1056 - mae: 0.2302 - val_loss: 0.1866 - val_mae: 0.3098
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 26s 14ms/step - loss: 0.1023 - mae: 0.2263 - val_loss: 0.1678 - val_mae: 0.2917
Epoch 9/10
1792/1792 ━━━

# GRU

In [36]:
def build_gru_model(sequence_length):

    model = Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, 1)
        ),

        GRU(64),

        Dense(64, activation="relu"),

        Dense(24)
    ])

    return model

In [37]:
gru_model = build_gru_model(
    SEQUENCE_LENGTH
)

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


history_gru = gru_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 31s 16ms/step - loss: 0.1941 - mae: 0.3221 - val_loss: 0.2191 - val_mae: 0.3415
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - loss: 0.1307 - mae: 0.2625 - val_loss: 0.2291 - val_mae: 0.3473
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.1222 - mae: 0.2520 - val_loss: 0.1924 - val_mae: 0.3185
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - loss: 0.1181 - mae: 0.2470 - val_loss: 0.2101 - val_mae: 0.3325
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - loss: 0.1139 - mae: 0.2417 - val_loss: 0.2007 - val_mae: 0.3236
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - loss: 0.1097 - mae: 0.2364 - val_loss: 0.2053 - val_mae: 0.3255
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.1058 - mae: 0.2312 - val_loss: 0.2021 - val_mae: 0.3180
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - loss: 0.1019 - mae: 0.2269 - val_loss: 0.1848 - val_mae: 0.3069
Epoch 9/10
1792/1792 ━━━

# LSTM + Bidirectional

In [38]:
def build_bilstm_model(sequence_length):

    model = Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, 1)
        ),

        Bidirectional(
            LSTM(64)
            
        ),

        Dense(64, activation="relu"),

        Dense(24)
    ])

    return model



In [39]:
bilstm_model = build_bilstm_model(
    SEQUENCE_LENGTH
)

bilstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


history_bilstm = bilstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 32s 16ms/step - loss: 0.1632 - mae: 0.2931 - val_loss: 0.1992 - val_mae: 0.3278
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 27s 15ms/step - loss: 0.1233 - mae: 0.2527 - val_loss: 0.1953 - val_mae: 0.3219
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 26s 14ms/step - loss: 0.1178 - mae: 0.2449 - val_loss: 0.2085 - val_mae: 0.3313
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 34s 19ms/step - loss: 0.1137 - mae: 0.2394 - val_loss: 0.1999 - val_mae: 0.3230
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 44s 24ms/step - loss: 0.1099 - mae: 0.2344 - val_loss: 0.1970 - val_mae: 0.3196
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 85s 26ms/step - loss: 0.1063 - mae: 0.2300 - val_loss: 0.1908 - val_mae: 0.3162
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 38s 21ms/step - loss: 0.1029 - mae: 0.2261 - val_loss: 0.1904 - val_mae: 0.3133
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 41s 21ms/step - loss: 0.0991 - mae: 0.2217 - val_loss: 0.1631 - val_mae: 0.2920
Epoch 9/10
1792/1792 ━━━

In [40]:
rnn_pred = rnn_model.predict(
    X_test
)

lstm_pred = lstm_model.predict(
    X_test
)

gru_pred = gru_model.predict(
    X_test
)

bilstm_pred = bilstm_model.predict(
    X_test
)

907/907 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step


In [41]:
rnn_pred

array([[-1.2305084 , -1.4759101 , -1.6879227 , ..., -0.27686167,
        -0.55407375, -0.8789704 ],
       [-1.4114001 , -1.5763494 , -1.729233  , ..., -0.49713024,
        -0.885509  , -1.216946  ],
       [-1.6371504 , -1.720455  , -1.7612306 , ..., -0.8673755 ,
        -1.1948874 , -1.4954021 ],
       ...,
       [-0.48177373, -0.4233473 , -0.2847467 , ..., -0.3153576 ,
        -0.41394132, -0.48677862],
       [-0.51784796, -0.36039233, -0.11612137, ..., -0.3906461 ,
        -0.52215624, -0.5581464 ],
       [-0.43894827, -0.28439015, -0.19987303, ..., -0.40226114,
        -0.45224366, -0.43975812]], shape=(29016, 24), dtype=float32)

In [42]:
lstm_pred

array([[-1.205038  , -1.4259329 , -1.5410649 , ..., -0.1923424 ,
        -0.42990133, -0.8005695 ],
       [-1.4112438 , -1.5131514 , -1.5476451 , ..., -0.44882873,
        -0.81731004, -1.1427656 ],
       [-1.5913804 , -1.6351677 , -1.6663904 , ..., -0.8112912 ,
        -1.1361617 , -1.3750821 ],
       ...,
       [-0.48629314, -0.44447953, -0.26639605, ..., -0.08732928,
        -0.07738775, -0.09098276],
       [-0.49576384, -0.39058796, -0.15768231, ..., -0.15957366,
        -0.16883506, -0.20822772],
       [-0.41666618, -0.2655816 , -0.11565696, ..., -0.23120408,
        -0.2699039 , -0.2794141 ]], shape=(29016, 24), dtype=float32)

In [43]:
gru_pred

array([[-1.1685225 , -1.4258811 , -1.5763818 , ..., -0.12011724,
        -0.3609775 , -0.7180393 ],
       [-1.3956922 , -1.5018405 , -1.5791193 , ..., -0.39447588,
        -0.76652855, -1.092772  ],
       [-1.4807659 , -1.563761  , -1.6008468 , ..., -0.8064635 ,
        -1.104887  , -1.3902807 ],
       ...,
       [-0.440309  , -0.5067245 , -0.4039201 , ..., -0.10444017,
        -0.14828931, -0.1470181 ],
       [-0.39435086, -0.26312685, -0.04484913, ..., -0.10694367,
        -0.16326466, -0.1789679 ],
       [-0.24428253, -0.04102907,  0.07074214, ..., -0.21822062,
        -0.30714825, -0.24352005]], shape=(29016, 24), dtype=float32)

In [44]:
bilstm_pred

array([[-1.19388306e+00, -1.50236344e+00, -1.65618980e+00, ...,
        -2.40557194e-01, -4.11176413e-01, -7.62770057e-01],
       [-1.51109564e+00, -1.65260315e+00, -1.73331511e+00, ...,
        -5.14990449e-01, -8.10904980e-01, -1.10437489e+00],
       [-1.60045612e+00, -1.71566451e+00, -1.73837769e+00, ...,
        -8.00282538e-01, -1.10428655e+00, -1.38378382e+00],
       ...,
       [-4.31446552e-01, -2.83181310e-01, -3.08422670e-02, ...,
         6.20874576e-02, -6.91485638e-03, -9.77833122e-02],
       [-3.53081465e-01, -1.26005158e-01,  7.02314079e-02, ...,
         1.20867044e-04, -7.86161497e-02, -1.71715409e-01],
       [-2.46434629e-01, -1.68803096e-01, -1.26743302e-01, ...,
        -1.62804395e-01, -1.86030895e-01, -1.77523240e-01]],
      shape=(29016, 24), dtype=float32)

# ReScaling

In [45]:
rnn_pred_original = scaler.inverse_transform(
    rnn_pred.reshape(-1, 1)
).reshape(rnn_pred.shape)

lstm_pred_original = scaler.inverse_transform(
    lstm_pred.reshape(-1, 1)
).reshape(lstm_pred.shape)

gru_pred_original = scaler.inverse_transform(
    gru_pred.reshape(-1, 1)
).reshape(gru_pred.shape)

bilstm_pred_original = scaler.inverse_transform(
    bilstm_pred.reshape(-1, 1)
).reshape(bilstm_pred.shape)

In [46]:
rnn_pred_original

array([[24235.438, 22640.723, 21262.984, ..., 30432.6  , 28631.168,
        26519.865],
       [23059.934, 21988.031, 20994.535, ..., 29001.209, 26477.375,
        24323.57 ],
       [21592.922, 21051.576, 20786.602, ..., 26595.215, 24466.916,
        22514.057],
       ...,
       [29101.002, 29480.68 , 30381.36 , ..., 30182.438, 29541.803,
        29068.479],
       [28866.578, 29889.785, 31477.15 , ..., 29693.184, 28838.58 ,
        28604.703],
       [29379.299, 30383.676, 30932.9  , ..., 29617.705, 29292.9  ,
        29374.035]], shape=(29016, 24), dtype=float32)

In [47]:
lstm_pred_original

array([[24400.953, 22965.494, 22217.322, ..., 30981.838, 29438.088,
        27029.346],
       [23060.95 , 22398.715, 22174.562, ..., 29315.092, 26920.559,
        24805.623],
       [21890.354, 21605.807, 21402.91 , ..., 26959.672, 24848.54 ,
        23295.941],
       ...,
       [29071.633, 29343.354, 30500.61 , ..., 31664.252, 31728.857,
        31640.512],
       [29010.09 , 29693.562, 31207.072, ..., 31194.781, 31134.598,
        30878.61 ],
       [29524.096, 30505.9  , 31480.168, ..., 30729.299, 30477.812,
        30416.012]], shape=(29016, 24), dtype=float32)

In [48]:
y_test_original = scaler.inverse_transform(
    y_test.reshape(-1, 1)
).reshape(y_test.shape)

# Error Calculation

In [49]:
def evaluate_deep_model(actual, predicted):

    actual_flat = actual.reshape(-1)
    predicted_flat = predicted.reshape(-1)

    mae = mean_absolute_error(
        actual_flat,
        predicted_flat
    )

    mse = mean_squared_error(
        actual_flat,
        predicted_flat
    )

    rmse = np.sqrt(mse)

    mape = mean_absolute_percentage_error(
        actual_flat,
        predicted_flat
    ) * 100

    r2 = r2_score(
        actual_flat,
        predicted_flat
    )

    bias = np.mean(
        predicted_flat - actual_flat
    )

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Bias": bias
    }

In [50]:
deep_results = {}

deep_results["RNN"] = evaluate_deep_model(
    y_test_original,
    rnn_pred_original
)

deep_results["LSTM"] = evaluate_deep_model(
    y_test_original,
    lstm_pred_original
)

deep_results["GRU"] = evaluate_deep_model(
    y_test_original,
    gru_pred_original
)

deep_results["Bi-LSTM"] = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original
)

In [51]:
deep_results

{'RNN': {'MAE': 2159.331354674648,
  'MSE': 9170441.803589357,
  'RMSE': np.float64(3028.2737332660927),
  'MAPE': 6.89319086863623,
  'R2': 0.7646875616342006,
  'Bias': np.float64(133.3896097106625)},
 'LSTM': {'MAE': 2249.314550682526,
  'MSE': 9890613.377197487,
  'RMSE': np.float64(3144.9345584920343),
  'MAPE': 7.189818537444811,
  'R2': 0.7462080453080479,
  'Bias': np.float64(204.11311603269533)},
 'GRU': {'MAE': 2269.806411696887,
  'MSE': 10129083.386320801,
  'RMSE': np.float64(3182.6220929165943),
  'MAPE': 7.284127932710614,
  'R2': 0.7400889334347291,
  'Bias': np.float64(298.0296915533043)},
 'Bi-LSTM': {'MAE': 2193.84026584953,
  'MSE': 9520878.386810856,
  'RMSE': np.float64(3085.5920642254146),
  'MAPE': 7.045962512775322,
  'R2': 0.7556954008793988,
  'Bias': np.float64(315.15977258562174)}}

In [52]:
deep_results_df = pd.DataFrame(
    deep_results
).T

deep_results_df

,MAE,MSE,RMSE,MAPE,R2,Bias
RNN,2159.331355,9.170442e+06,3028.273733,6.893191,0.764688,133.389610
LSTM,2249.314551,9.890613e+06,3144.934558,7.189819,0.746208,204.113116
GRU,2269.806412,1.012908e+07,3182.622093,7.284128,0.740089,298.029692
Bi-LSTM,2193.840266,9.520878e+06,3085.592064,7.045963,0.755695,315.159773


# 48

In [53]:
SEQUENCE_LENGTH = 48
FORECAST_HORIZON = 24


X_train, y_train = create_sequences(
    train_scaled,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [54]:
X_train

array([[[-1.61544069],
        [-1.73285452],
        [-1.77132563],
        ...,
        [-0.34758674],
        [-0.74537803],
        [-1.1354751 ]],

       [[-1.73285452],
        [-1.77132563],
        [-1.76363141],
        ...,
        [-0.74537803],
        [-1.1354751 ],
        [-1.40738892]],

       [[-1.77132563],
        [-1.76363141],
        [-1.67699446],
        ...,
        [-1.1354751 ],
        [-1.40738892],
        [-1.55465633]],

       ...,

       [[ 0.59433995],
        [ 0.1911627 ],
        [-0.24833127],
        ...,
        [ 1.17617704],
        [ 0.90672537],
        [ 0.71883247]],

       [[ 0.1911627 ],
        [-0.24833127],
        [-0.35081831],
        ...,
        [ 0.90672537],
        [ 0.71883247],
        [ 0.55879264]],

       [[-0.24833127],
        [-0.35081831],
        [-0.66135712],
        ...,
        [ 0.71883247],
        [ 0.55879264],
        [ 0.18916221]]], shape=(114648, 48, 1))

In [55]:
y_train

array([[[-1.40738892],
        [-1.55465633],
        [-1.6192878 ],
        ...,
        [-0.21786215],
        [-0.57810564],
        [-0.93250152]],

       [[-1.55465633],
        [-1.6192878 ],
        [-1.64267823],
        ...,
        [-0.57810564],
        [-0.93250152],
        [-1.18087101]],

       [[-1.6192878 ],
        [-1.64267823],
        [-1.58112446],
        ...,
        [-0.93250152],
        [-1.18087101],
        [-1.29105228]],

       ...,

       [[ 0.55879264],
        [ 0.18916221],
        [-0.23278894],
        ...,
        [ 0.68851723],
        [ 0.49893159],
        [ 0.73114322]],

       [[ 0.18916221],
        [-0.23278894],
        [-0.634889  ],
        ...,
        [ 0.49893159],
        [ 0.73114322],
        [ 0.55817711]],

       [[-0.23278894],
        [-0.634889  ],
        [-0.90634116],
        ...,
        [ 0.73114322],
        [ 0.55817711],
        [ 0.16961888]]], shape=(114648, 24, 1))

In [56]:
val_input = np.concatenate([
    train_scaled[-SEQUENCE_LENGTH:],
    val_scaled
])


X_val, y_val = create_sequences(
    val_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [57]:
val_input

array([[-0.40806333],
       [-0.4776191 ],
       [-0.7476863 ],
       ...,
       [-0.26341195],
       [-0.4983935 ],
       [-0.80985561]], shape=(1488, 1))

In [58]:
test_input = np.concatenate([
    val_scaled[-SEQUENCE_LENGTH:],
    test_scaled
])


X_test, y_test = create_sequences(
    test_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

# Basic RNN with 48

In [59]:
rnn_model = build_rnn_model(
    SEQUENCE_LENGTH
)

rnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

history_rnn = rnn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 21s 10ms/step - loss: 0.1566 - mae: 0.2859 - val_loss: 0.2375 - val_mae: 0.3522
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.1199 - mae: 0.2505 - val_loss: 0.2254 - val_mae: 0.3415
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.1066 - mae: 0.2355 - val_loss: 0.2409 - val_mae: 0.3537
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.0976 - mae: 0.2247 - val_loss: 0.1905 - val_mae: 0.3167
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - loss: 0.0939 - mae: 0.2203 - val_loss: 0.1993 - val_mae: 0.3222
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.0902 - mae: 0.2153 - val_loss: 0.1549 - val_mae: 0.2851
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.0880 - mae: 0.2126 - val_loss: 0.2251 - val_mae: 0.3277
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.0862 - mae: 0.2101 - val_loss: 0.1586 - val_mae: 0.2890
Epoch 9/10
1792/1792 ━━━

# LSTM 48

In [60]:
lstm_model = build_lstm_model(
    SEQUENCE_LENGTH
)

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

history_lstm = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 48s 26ms/step - loss: 0.1721 - mae: 0.3030 - val_loss: 0.2210 - val_mae: 0.3469
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 45s 25ms/step - loss: 0.1200 - mae: 0.2505 - val_loss: 0.2009 - val_mae: 0.3272
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 45s 25ms/step - loss: 0.1105 - mae: 0.2381 - val_loss: 0.1933 - val_mae: 0.3141
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 56s 31ms/step - loss: 0.1006 - mae: 0.2264 - val_loss: 0.2112 - val_mae: 0.3279
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 52s 29ms/step - loss: 0.0937 - mae: 0.2180 - val_loss: 0.1583 - val_mae: 0.2874
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 51s 28ms/step - loss: 0.0885 - mae: 0.2118 - val_loss: 0.1896 - val_mae: 0.2972
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 46s 26ms/step - loss: 0.0844 - mae: 0.2065 - val_loss: 0.1669 - val_mae: 0.2765
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 80s 25ms/step - loss: 0.0815 - mae: 0.2029 - val_loss: 0.1504 - val_mae: 0.2719
Epoch 9/10
1792/1792 ━━━

# GRU 48

In [61]:
gru_model = build_gru_model(
    SEQUENCE_LENGTH
)

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


history_gru = gru_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 52s 28ms/step - loss: 0.1886 - mae: 0.3167 - val_loss: 0.2838 - val_mae: 0.3900
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 48s 27ms/step - loss: 0.1258 - mae: 0.2577 - val_loss: 0.2379 - val_mae: 0.3503
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 49s 27ms/step - loss: 0.1167 - mae: 0.2465 - val_loss: 0.2404 - val_mae: 0.3613
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 49s 27ms/step - loss: 0.1085 - mae: 0.2369 - val_loss: 0.1729 - val_mae: 0.2985
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 57s 32ms/step - loss: 0.0979 - mae: 0.2245 - val_loss: 0.1661 - val_mae: 0.2902
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 51s 28ms/step - loss: 0.0903 - mae: 0.2152 - val_loss: 0.1969 - val_mae: 0.3046
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 49s 27ms/step - loss: 0.0861 - mae: 0.2093 - val_loss: 0.1691 - val_mae: 0.2931
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 50s 28ms/step - loss: 0.0828 - mae: 0.2050 - val_loss: 0.1506 - val_mae: 0.2738
Epoch 9/10
1792/1792 ━━━

# BI-directional 48

In [62]:
bilstm_model = build_bilstm_model(
    SEQUENCE_LENGTH
)

bilstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


history_bilstm = bilstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 57s 30ms/step - loss: 0.1604 - mae: 0.2926 - val_loss: 0.2199 - val_mae: 0.3476
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 52s 29ms/step - loss: 0.1185 - mae: 0.2489 - val_loss: 0.2119 - val_mae: 0.3382
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 52s 29ms/step - loss: 0.1074 - mae: 0.2346 - val_loss: 0.2149 - val_mae: 0.3394
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 52s 29ms/step - loss: 0.0986 - mae: 0.2236 - val_loss: 0.2290 - val_mae: 0.3503
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 52s 29ms/step - loss: 0.0910 - mae: 0.2146 - val_loss: 0.1679 - val_mae: 0.2997
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 53s 30ms/step - loss: 0.0857 - mae: 0.2081 - val_loss: 0.1387 - val_mae: 0.2677
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 66s 37ms/step - loss: 0.0816 - mae: 0.2027 - val_loss: 0.1273 - val_mae: 0.2557
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 53s 30ms/step - loss: 0.0787 - mae: 0.1992 - val_loss: 0.1276 - val_mae: 0.2557
Epoch 9/10
1792/1792 ━━━

In [63]:
rnn_pred_48 = rnn_model.predict(
    X_test
)

lstm_pred_48 = lstm_model.predict(
    X_test
)

gru_pred_48 = gru_model.predict(
    X_test
)

bilstm_pred_48 = bilstm_model.predict(
    X_test
)

907/907 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step


In [64]:
rnn_pred_48

array([[-1.1884936 , -1.4326768 , -1.5802344 , ..., -0.43170497,
        -0.6491433 , -0.93388665],
       [-1.4230045 , -1.5515798 , -1.6265591 , ..., -0.6280364 ,
        -0.9775323 , -1.32473   ],
       [-1.5741282 , -1.6346339 , -1.6994817 , ..., -0.8425374 ,
        -1.1825141 , -1.4602616 ],
       ...,
       [-0.4552466 , -0.46589544, -0.31373888, ..., -0.22040132,
        -0.2460406 , -0.2553158 ],
       [-0.3660752 , -0.15856647,  0.1012245 , ..., -0.0177917 ,
        -0.03290451, -0.01880637],
       [-0.12754546,  0.16570899,  0.24335869, ..., -0.07670957,
        -0.04914607, -0.06873518]], shape=(29016, 24), dtype=float32)

In [65]:
lstm_pred_48

array([[-1.2200806 , -1.5050797 , -1.6817259 , ..., -0.32416615,
        -0.5309    , -0.8289598 ],
       [-1.4775597 , -1.6319125 , -1.7194424 , ..., -0.46592292,
        -0.82339394, -1.191193  ],
       [-1.6165274 , -1.7450099 , -1.7863495 , ..., -0.81014466,
        -1.1862303 , -1.4488418 ],
       ...,
       [-0.43775284, -0.47339892, -0.38578478, ..., -0.23284747,
        -0.23552151, -0.22481097],
       [-0.41678008, -0.31124195, -0.09135551, ..., -0.2672163 ,
        -0.2878065 , -0.32316506],
       [-0.21959804, -0.02455736,  0.1723038 , ..., -0.33099258,
        -0.36339498, -0.350597  ]], shape=(29016, 24), dtype=float32)

In [66]:
gru_pred_48

array([[-1.2200534 , -1.493103  , -1.6515559 , ..., -0.18039104,
        -0.42431232, -0.7771851 ],
       [-1.4355069 , -1.5615613 , -1.6554829 , ..., -0.39279008,
        -0.76306635, -1.080266  ],
       [-1.6315441 , -1.6918197 , -1.7012118 , ..., -0.7683493 ,
        -1.1543164 , -1.3966556 ],
       ...,
       [-0.47742963, -0.480645  , -0.34524262, ..., -0.1561274 ,
        -0.17854172, -0.24686186],
       [-0.42372423, -0.33637524, -0.15174168, ..., -0.19789097,
        -0.2472177 , -0.28696585],
       [-0.2518718 , -0.04386647,  0.13120209, ..., -0.23072413,
        -0.28005308, -0.26116312]], shape=(29016, 24), dtype=float32)

In [67]:
bilstm_pred_48

array([[-1.1973019 , -1.5219115 , -1.6682297 , ..., -0.26599392,
        -0.47701573, -0.8185954 ],
       [-1.4498402 , -1.6452626 , -1.7374276 , ..., -0.48395866,
        -0.82545567, -1.1850774 ],
       [-1.6396471 , -1.7353646 , -1.7283281 , ..., -0.83433646,
        -1.1806034 , -1.4958509 ],
       ...,
       [-0.48642454, -0.46062925, -0.35589093, ..., -0.17576763,
        -0.3101005 , -0.42186832],
       [-0.46163675, -0.30021733, -0.07432855, ..., -0.17409916,
        -0.31517208, -0.38870984],
       [-0.26964632,  0.01186168,  0.23995702, ..., -0.1989794 ,
        -0.31610703, -0.31860402]], shape=(29016, 24), dtype=float32)

In [68]:
def evaluate_deep_model(actual, predicted):

    actual_flat = actual.reshape(-1)
    predicted_flat = predicted.reshape(-1)

    mae = mean_absolute_error(
        actual_flat,
        predicted_flat
    )

    mse = mean_squared_error(
        actual_flat,
        predicted_flat
    )

    rmse = np.sqrt(mse)

    mape = mean_absolute_percentage_error(
        actual_flat,
        predicted_flat
    ) * 100

    r2 = r2_score(
        actual_flat,
        predicted_flat
    )

    bias = np.mean(
        predicted_flat - actual_flat
    )

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Bias": bias
    }

In [69]:
rnn_pred_original_48 = scaler.inverse_transform(
    rnn_pred_48.reshape(-1, 1)
).reshape(rnn_pred_48.shape)

lstm_pred_original_48    = scaler.inverse_transform(
    lstm_pred_48.reshape(-1, 1)
).reshape(lstm_pred_48.shape)

gru_pred_original_48 = scaler.inverse_transform(
    gru_pred_48.reshape(-1, 1)
).reshape(gru_pred_48.shape)

bilstm_pred_original_48 = scaler.inverse_transform(
    bilstm_pred_48.reshape(-1, 1)
).reshape(bilstm_pred_48.shape)

In [70]:
deep_results = {}

deep_results["RNN"] = evaluate_deep_model(
    y_test_original,
    rnn_pred_original_48
)

deep_results["LSTM"] = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_48
)

deep_results["GRU"] = evaluate_deep_model(
    y_test_original,
    gru_pred_original_48
)

deep_results["Bi-LSTM"] = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

In [71]:
deep_results

{'RNN': {'MAE': 2291.5635493295767,
  'MSE': 10429830.554360397,
  'RMSE': np.float64(3229.524818663018),
  'MAPE': 7.312749040902447,
  'R2': 0.7323717971223536,
  'Bias': np.float64(114.41555539883078)},
 'LSTM': {'MAE': 2138.12304464529,
  'MSE': 9273593.057611093,
  'RMSE': np.float64(3045.2574698391422),
  'MAPE': 6.746704323496684,
  'R2': 0.7620407128100967,
  'Bias': np.float64(32.82833107849351)},
 'GRU': {'MAE': 2121.910230768591,
  'MSE': 9279466.472572394,
  'RMSE': np.float64(3046.2216716076973),
  'MAPE': 6.721053858730308,
  'R2': 0.7618900016856297,
  'Bias': np.float64(144.7361138947428)},
 'Bi-LSTM': {'MAE': 2153.0383143635777,
  'MSE': 9328833.780914746,
  'RMSE': np.float64(3054.313962400517),
  'MAPE': 6.844891009195177,
  'R2': 0.7606232424661286,
  'Bias': np.float64(186.55001207296817)}}

In [72]:
deep_results_df_48 = pd.DataFrame(
    deep_results
).T

deep_results_df_48

,MAE,MSE,RMSE,MAPE,R2,Bias
RNN,2291.563549,1.042983e+07,3229.524819,7.312749,0.732372,114.415555
LSTM,2138.123045,9.273593e+06,3045.257470,6.746704,0.762041,32.828331
GRU,2121.910231,9.279466e+06,3046.221672,6.721054,0.761890,144.736114
Bi-LSTM,2153.038314,9.328834e+06,3054.313962,6.844891,0.760623,186.550012


# 168

In [73]:
SEQUENCE_LENGTH = 168
FORECAST_HORIZON = 24


X_train, y_train = create_sequences(
    train_scaled,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [74]:
val_input = np.concatenate([
    train_scaled[-SEQUENCE_LENGTH:],
    val_scaled
])


X_val, y_val = create_sequences(
    val_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [75]:
test_input = np.concatenate([
    val_scaled[-SEQUENCE_LENGTH:],
    test_scaled
])


X_test, y_test = create_sequences(
    test_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

# Basic RNN 168

In [76]:
rnn_model = build_rnn_model(
    SEQUENCE_LENGTH
)

rnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

history_rnn = rnn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10


W0000 00:00:1786427618.864976   93018 cpu_allocator_impl.cc:82] Allocation of 76962816 exceeds 10% of free system memory.


1790/1790 ━━━━━━━━━━━━━━━━━━━━ 50s 27ms/step - loss: 0.1553 - mae: 0.2860 - val_loss: 0.2578 - val_mae: 0.3760
Epoch 2/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 48s 27ms/step - loss: 0.1138 - mae: 0.2448 - val_loss: 0.2105 - val_mae: 0.3405
Epoch 3/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 48s 27ms/step - loss: 0.1006 - mae: 0.2287 - val_loss: 0.1841 - val_mae: 0.3068
Epoch 4/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 48s 27ms/step - loss: 0.0955 - mae: 0.2223 - val_loss: 0.1573 - val_mae: 0.2865
Epoch 5/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 49s 27ms/step - loss: 0.0912 - mae: 0.2170 - val_loss: 0.1610 - val_mae: 0.2962
Epoch 6/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 50s 28ms/step - loss: 0.0872 - mae: 0.2119 - val_loss: 0.1547 - val_mae: 0.2863
Epoch 7/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 49s 28ms/step - loss: 0.0862 - mae: 0.2104 - val_loss: 0.1499 - val_mae: 0.2796
Epoch 8/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 60s 34ms/step - loss: 0.0849 - mae: 0.2087 - val_loss: 0.1564 - val_mae: 0.2864
Epoch 9/10
1790/1790 ━━━━━━━━━━━━━━

# LSTM 168

In [77]:
lstm_model = build_lstm_model(
    SEQUENCE_LENGTH
)

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

history_lstm = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
)

Epoch 1/10


W0000 00:00:1786428140.535812   93018 cpu_allocator_impl.cc:82] Allocation of 76962816 exceeds 10% of free system memory.


1790/1790 ━━━━━━━━━━━━━━━━━━━━ 150s 82ms/step - loss: 0.1673 - mae: 0.2980 - val_loss: 0.3007 - val_mae: 0.4095
Epoch 2/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 147s 82ms/step - loss: 0.1113 - mae: 0.2418 - val_loss: 0.2262 - val_mae: 0.3515
Epoch 3/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 163s 91ms/step - loss: 0.0967 - mae: 0.2227 - val_loss: 0.2156 - val_mae: 0.3365
Epoch 4/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 168s 94ms/step - loss: 0.0875 - mae: 0.2102 - val_loss: 0.1646 - val_mae: 0.2862
Epoch 5/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 175s 98ms/step - loss: 0.0827 - mae: 0.2035 - val_loss: 0.1290 - val_mae: 0.2520
Epoch 6/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 151s 84ms/step - loss: 0.0778 - mae: 0.1969 - val_loss: 0.1476 - val_mae: 0.2761
Epoch 7/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 150s 84ms/step - loss: 0.0744 - mae: 0.1921 - val_loss: 0.1179 - val_mae: 0.2409
Epoch 8/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 143s 80ms/step - loss: 0.0790 - mae: 0.1971 - val_loss: 0.1333 - val_mae: 0.2561
Epoch 9/10
1790/1790 ━━━━━━

# GRU 168

In [78]:
gru_model = build_gru_model(
    SEQUENCE_LENGTH
)

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


history_gru = gru_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10


W0000 00:00:1786429724.719530   93018 cpu_allocator_impl.cc:82] Allocation of 76962816 exceeds 10% of free system memory.


1790/1790 ━━━━━━━━━━━━━━━━━━━━ 242s 133ms/step - loss: 0.1908 - mae: 0.3197 - val_loss: 0.2335 - val_mae: 0.3457
Epoch 2/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 216s 107ms/step - loss: 0.1220 - mae: 0.2549 - val_loss: 0.2330 - val_mae: 0.3485
Epoch 3/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 177s 99ms/step - loss: 0.1079 - mae: 0.2390 - val_loss: 0.2139 - val_mae: 0.3358
Epoch 4/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 197s 110ms/step - loss: 0.0974 - mae: 0.2258 - val_loss: 0.1829 - val_mae: 0.3150
Epoch 5/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 177s 99ms/step - loss: 0.0890 - mae: 0.2144 - val_loss: 0.1506 - val_mae: 0.2812
Epoch 6/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 187s 104ms/step - loss: 0.0824 - mae: 0.2059 - val_loss: 0.1347 - val_mae: 0.2643
Epoch 7/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 219s 122ms/step - loss: 0.0773 - mae: 0.1993 - val_loss: 0.1423 - val_mae: 0.2716
Epoch 8/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 194s 109ms/step - loss: 0.0742 - mae: 0.1949 - val_loss: 0.1297 - val_mae: 0.2571
Epoch 9/10
1790/1790 

# Bidirectional LSTM 168

In [79]:
bilstm_model = build_bilstm_model(
    SEQUENCE_LENGTH
)

bilstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


history_bilstm = bilstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10


W0000 00:00:1786431735.044754   93018 cpu_allocator_impl.cc:82] Allocation of 76962816 exceeds 10% of free system memory.


1790/1790 ━━━━━━━━━━━━━━━━━━━━ 220s 120ms/step - loss: 0.1386 - mae: 0.2692 - val_loss: 0.1696 - val_mae: 0.3075
Epoch 2/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 284s 158ms/step - loss: 0.0968 - mae: 0.2250 - val_loss: 0.1518 - val_mae: 0.2850
Epoch 3/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 303s 169ms/step - loss: 0.0876 - mae: 0.2113 - val_loss: 0.1500 - val_mae: 0.2850
Epoch 4/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 467s 250ms/step - loss: 0.0825 - mae: 0.2041 - val_loss: 0.1361 - val_mae: 0.2662
Epoch 5/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 484s 270ms/step - loss: 0.0781 - mae: 0.1980 - val_loss: 0.1235 - val_mae: 0.2532
Epoch 6/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 303s 169ms/step - loss: 0.0745 - mae: 0.1930 - val_loss: 0.1238 - val_mae: 0.2515
Epoch 7/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 354s 187ms/step - loss: 0.0712 - mae: 0.1885 - val_loss: 0.1168 - val_mae: 0.2495
Epoch 8/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 4370s 2s/step - loss: 0.0679 - mae: 0.1842 - val_loss: 0.1567 - val_mae: 0.2859
Epoch 9/10
1790/1790 

In [80]:
rnn_pred_168 = rnn_model.predict(
    X_test
)

lstm_pred_168 = lstm_model.predict(
    X_test
)

gru_pred_168 = gru_model.predict(
    X_test
)

bilstm_pred_168 = bilstm_model.predict(
    X_test
)

907/907 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 17s 18ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 25s 27ms/step


In [81]:
def evaluate_deep_model(actual, predicted):

    actual_flat = actual.reshape(-1)
    predicted_flat = predicted.reshape(-1)

    mae = mean_absolute_error(
        actual_flat,
        predicted_flat
    )

    mse = mean_squared_error(
        actual_flat,
        predicted_flat
    )

    rmse = np.sqrt(mse)

    mape = mean_absolute_percentage_error(
        actual_flat,
        predicted_flat
    ) * 100

    r2 = r2_score(
        actual_flat,
        predicted_flat
    )

    bias = np.mean(
        predicted_flat - actual_flat
    )

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Bias": bias
    }

In [82]:
rnn_pred_original_168 = scaler.inverse_transform(
    rnn_pred_168.reshape(-1, 1)
).reshape(rnn_pred_168.shape)

lstm_pred_original_168 = scaler.inverse_transform(
    lstm_pred_168.reshape(-1, 1)
).reshape(lstm_pred_168.shape)

gru_pred_original_168 = scaler.inverse_transform(
    gru_pred_168.reshape(-1, 1)
).reshape(gru_pred_168.shape)

bilstm_pred_original_168 = scaler.inverse_transform(
    bilstm_pred_168.reshape(-1, 1)
).reshape(bilstm_pred_168.shape)

In [83]:
deep_results = {}

deep_results["RNN"] = evaluate_deep_model(
    y_test_original,
    rnn_pred_original_168
)

deep_results["LSTM"] = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

deep_results["GRU"] = evaluate_deep_model(
    y_test_original,
    gru_pred_original_168
)

deep_results["Bi-LSTM"] = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_168
)

In [84]:
deep_results

{'RNN': {'MAE': 2329.7542053817247,
  'MSE': 10735139.546826636,
  'RMSE': np.float64(3276.4522805660754),
  'MAPE': 7.514981085759248,
  'R2': 0.7245376049415455,
  'Bias': np.float64(438.5619373249664)},
 'LSTM': {'MAE': 1992.7002130968915,
  'MSE': 8144207.225169442,
  'RMSE': np.float64(2853.805744119498),
  'MAPE': 6.2419873802107375,
  'R2': 0.7910206180076427,
  'Bias': np.float64(102.4336904352744)},
 'GRU': {'MAE': 2121.4068120271554,
  'MSE': 9012742.46486139,
  'RMSE': np.float64(3002.12299296038),
  'MAPE': 6.736924981352306,
  'R2': 0.768734107778818,
  'Bias': np.float64(258.7834736575384)},
 'Bi-LSTM': {'MAE': 2004.840544291229,
  'MSE': 8069632.4762456585,
  'RMSE': np.float64(2840.7098542874205),
  'MAPE': 6.277558344741292,
  'R2': 0.7929341971334494,
  'Bias': np.float64(-58.13798549733786)}}

In [ ]:
deep_results_df_168 = pd.DataFrame(
    deep_results
).T

deep_results_df_168


,MAE,MSE,RMSE,MAPE,R2,Bias
RNN,2329.754205,1.073514e+07,3276.452281,7.514981,0.724538,438.561937
LSTM,1992.700213,8.144207e+06,2853.805744,6.241987,0.791021,102.433690
GRU,2121.406812,9.012742e+06,3002.122993,6.736925,0.768734,258.783474
Bi-LSTM,2004.840544,8.069632e+06,2840.709854,6.277558,0.792934,-58.137985


In [88]:
bilstm_model.save(
    "./Models/bilstm_model_168_79.keras",
)

In [90]:
import pandas as pd

data = {
    "Sequence": [
        24, 24, 24, 24,
        48, 48, 48, 48,
        168, 168, 168, 168
    ],
    "Model": [
        "RNN", "LSTM", "GRU", "Bi-LSTM",
        "RNN", "LSTM", "GRU", "Bi-LSTM",
        "RNN", "LSTM", "GRU", "Bi-LSTM"
    ],
    "MAE": [
        2159.331355,
        2249.314551,
        2269.806412,
        2193.840266,
        2291.563549,
        2138.123045,
        2121.910231,
        2153.038314,
        2329.754205,
        1992.700213,
        2121.406812,
        2004.840544
    ],
    "MSE": [
        9.170442e+06,
        9.890613e+06,
        1.012908e+07,
        9.520878e+06,
        1.042983e+07,
        9.273593e+06,
        9.279466e+06,
        9.328834e+06,
        1.073514e+07,
        8.144207e+06,
        9.012742e+06,
        8.069632e+06
    ],
    "RMSE": [
        3028.273733,
        3144.934558,
        3182.622093,
        3085.592064,
        3229.524819,
        3045.257470,
        3046.221672,
        3054.313962,
        3276.452281,
        2853.805744,
        3002.122993,
        2840.709854
    ],
    "MAPE": [
        6.893191,
        7.189819,
        7.284128,
        7.045963,
        7.312749,
        6.746704,
        6.721054,
        6.844891,
        7.514981,
        6.241987,
        6.736925,
        6.277558
    ],
    "R2": [
        0.764688,
        0.746208,
        0.740089,
        0.755695,
        0.732372,
        0.762041,
        0.761890,
        0.760623,
        0.724538,
        0.791021,
        0.768734,
        0.792934
    ],
    "Bias": [
        133.389610,
        204.113116,
        298.029692,
        315.159773,
        114.415555,
        32.828331,
        144.736114,
        186.550012,
        438.561937,
        102.433690,
        258.783474,
        -58.137985
    ]
}

df = pd.DataFrame(data)

df

,Sequence,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,24,RNN,2159.331355,9170442.0,3028.273733,6.893191,0.764688,133.389610
1,24,LSTM,2249.314551,9890613.0,3144.934558,7.189819,0.746208,204.113116
2,24,GRU,2269.806412,10129080.0,3182.622093,7.284128,0.740089,298.029692
3,24,Bi-LSTM,2193.840266,9520878.0,3085.592064,7.045963,0.755695,315.159773
4,48,RNN,2291.563549,10429830.0,3229.524819,7.312749,0.732372,114.415555
5,48,LSTM,2138.123045,9273593.0,3045.257470,6.746704,0.762041,32.828331
6,48,GRU,2121.910231,9279466.0,3046.221672,6.721054,0.761890,144.736114
7,48,Bi-LSTM,2153.038314,9328834.0,3054.313962,6.844891,0.760623,186.550012
8,168,RNN,2329.754205,10735140.0,3276.452281,7.514981,0.724538,438.561937
9,168,LSTM,1992.700213,8144207.0,2853.805744,6.241987,0.791021,102.433690


In [91]:
df.sort_values(by=["R2"], ascending=False)

,Sequence,Model,MAE,MSE,RMSE,MAPE,R2,Bias
11,168,Bi-LSTM,2004.840544,8069632.0,2840.709854,6.277558,0.792934,-58.137985
9,168,LSTM,1992.700213,8144207.0,2853.805744,6.241987,0.791021,102.433690
10,168,GRU,2121.406812,9012742.0,3002.122993,6.736925,0.768734,258.783474
0,24,RNN,2159.331355,9170442.0,3028.273733,6.893191,0.764688,133.389610
5,48,LSTM,2138.123045,9273593.0,3045.257470,6.746704,0.762041,32.828331
6,48,GRU,2121.910231,9279466.0,3046.221672,6.721054,0.761890,144.736114
7,48,Bi-LSTM,2153.038314,9328834.0,3054.313962,6.844891,0.760623,186.550012
3,24,Bi-LSTM,2193.840266,9520878.0,3085.592064,7.045963,0.755695,315.159773
1,24,LSTM,2249.314551,9890613.0,3144.934558,7.189819,0.746208,204.113116
2,24,GRU,2269.806412,10129080.0,3182.622093,7.284128,0.740089,298.029692
